# 전자상거래 데이터 모델링과 집계

AI 보조 작성 코드입니다. 아래 결과는 실행 스크립트 또는 VS Code의 실제 TypeScript 커널 실행으로 저장합니다.


## 1. 모델과 샘플 데이터
`type`은 허용하는 문자열 집합, `interface`는 객체의 필드 구조를 정의합니다. `Identifiable`의 id를 공통 상속합니다.


In [1]:
interface Identifiable { id: number; }
type UserRole = 'admin' | 'customer' | 'guest';
interface User extends Identifiable { name: string; email: string; role: UserRole; isActive: boolean; }
type Category = 'electronics' | 'clothing' | 'books';
interface Product extends Identifiable { title: string; price: number; category: Category; stock: number; discountRate?: number; }
const users: User[] = [
 {id:1,name:'Kim',email:'kim@example.com',role:'admin',isActive:true},
 {id:2,name:'Lee',email:'lee@example.com',role:'customer',isActive:true},
 {id:3,name:'Park',email:'park@example.com',role:'guest',isActive:false}
];
const products: Product[] = [
 {id:1,title:'Slacks',price:40000,category:'clothing',stock:10,discountRate:0.1},
 {id:2,title:'Book',price:20000,category:'books',stock:4},
 {id:3,title:'Keyboard',price:50000,category:'electronics',stock:2,discountRate:0}
];
console.log('회원',users.length,'상품',products.length);


회원 3 상품 3


## 2. 필터와 결제 금액
`filter`는 원본 배열을 바꾸지 않습니다. `discountRate ?? 0`은 생략한 할인율만 0으로 처리합니다. 할인율은 0~1, 금액은 원 단위 반올림이며 각 상품 한 개의 금액을 합산합니다. 재고 수량은 결제 수량이 아닙니다.


In [2]:
function filterUsersByRole(users: User[], targetRole: UserRole): User[] {
 return users.filter(user => user.role === targetRole);
}
function calculateTotalPrice(products: Product[]): number {
 return products.reduce((total, product) => {
  const rate = product.discountRate ?? 0;
  if (!Number.isFinite(rate) || rate<0 || rate>1 || !Number.isFinite(product.price) || product.price<0) throw new RangeError('잘못된 가격/할인율');
  return total + Math.round(product.price * (1-rate));
 }, 0);
}
console.log(filterUsersByRole(users,'customer'));
console.log('총액',calculateTotalPrice(products));
if(calculateTotalPrice(products)!==106000) throw new Error('가격 계산 실패');
if(calculateTotalPrice([])!==0) throw new Error('빈 배열 처리 실패');
console.log('금액 및 빈 배열 검증 통과');


[
  {
    id: 2,
    name: 'Lee',
    email: 'lee@example.com',
    role: 'customer',
    isActive: true
  }
]
총액 106000
금액 및 빈 배열 검증 통과


## 3. 경계값 검증
선택 필드 생략·0·1·오류 입력을 검사합니다. 기대값과 다르면 예외를 발생시키므로 PASS 출력만으로 검증하지 않습니다.


In [3]:
// 할인율 미입력, 0%, 100%와 빈 결과를 각각 검증합니다.
const plain: Product = { id: 10, title: 'Plain', price: 10000, category: 'clothing', stock: 0 };
if (calculateTotalPrice([plain]) !== 10000) throw new Error('정가 처리 실패');
if (calculateTotalPrice([{ ...plain, discountRate: 0 }]) !== 10000) throw new Error('0% 할인 처리 실패');
if (calculateTotalPrice([{ ...plain, discountRate: 1 }]) !== 0) throw new Error('100% 할인 처리 실패');
if (filterUsersByRole([], 'admin').length !== 0) throw new Error('빈 회원 목록 처리 실패');
for (const rate of [-0.1, 1.1, Number.NaN]) {
 let rejected = false;
 try { calculateTotalPrice([{ ...plain, discountRate: rate }]); }
 catch (error: unknown) { rejected = error instanceof RangeError; }
 if (!rejected) throw new Error('잘못된 할인율이 통과함');
}
console.log('PASS: 할인율 미입력 / 0% / 100% / 빈 목록 / 잘못된 할인율 3종');


PASS: 할인율 미입력 / 0% / 100% / 빈 목록 / 잘못된 할인율 3종
